# Camera capture experiment

Standalone notebook for testing and tuning camera-based tag capture: take a photo with this machine's webcam, see exactly how the quality gate scores it, and run it through the real pipeline -- all with fast iteration, without needing to launch the full Streamlit app each time.

**Kernel:** select "Python (igi-ocr)".

**Important caveat:** this notebook captures via OpenCV's `cv2.VideoCapture`, talking directly to whatever camera is attached to *this machine*. The deployed app instead uses `st.camera_input()`, which captures through the *browser's* camera API (`getUserMedia`) -- on a phone, through Windows 11 Phone Link, or through a different webcam driver, characteristics (default resolution, auto-exposure/focus behavior, compression) can all differ from what you see here. Treat this as a tool for understanding capture quality and tuning `quality.py`'s thresholds and `parsing.py`'s tolerance -- not as a pixel-perfect simulation of the deployed capture path.

**One concrete finding from building this notebook, already fixed in `app.py`:** `st.camera_input()`'s default resolution is tied to the *widget's on-screen display size* -- so shrinking the widget visually (done earlier, to make the live preview smaller) was also silently capturing lower-resolution photos. Fixed by passing `resolution="1080p"` explicitly, which requests a fixed target resolution independent of how large the widget is drawn.

## Setup

In [ ]:
import sys
import time

sys.path.insert(0, "..")  # project root, so ocr/imaging/parsing/quality/pipeline import correctly
# (Jupyter's default working directory is this notebook's own folder)

import cv2
import matplotlib.pyplot as plt

import ocr
import pipeline
import quality

CAMERA_INDEX = 0  # change if you have more than one camera and want a different one

## What resolutions does this camera actually support?

`cv2.VideoCapture`'s default resolution is often much lower than the camera's real capability (commonly 640x480) unless you explicitly request higher. This cell requests a few common resolutions and reports what the driver actually returns -- run it once to know your camera's real ceiling before capturing.

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
print("default:", cap.get(cv2.CAP_PROP_FRAME_WIDTH), "x", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

for w, h in [(1280, 720), (1920, 1080), (3840, 2160)]:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)
    actual_w, actual_h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    print(f"requested {w}x{h} -> got {actual_w:.0f}x{actual_h:.0f}")

cap.release()

## Capture a photo

Position a real IGI tag in front of the camera, then run this cell. Re-run it any time to grab a fresh frame (e.g. after adjusting lighting/angle/distance). `CAPTURE_WIDTH`/`CAPTURE_HEIGHT` request the camera's best resolution found above.

In [ ]:
CAPTURE_WIDTH, CAPTURE_HEIGHT = 1280, 720  # set to whatever this camera actually supports, from the cell above

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAPTURE_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAPTURE_HEIGHT)

# Several webcams return a stale/black first frame or two right after opening
# or changing resolution -- warm up briefly before keeping a frame.
for _ in range(5):
    ret, frame = cap.read()
    time.sleep(0.1)
cap.release()

assert ret, "Failed to read from the camera -- check CAMERA_INDEX"
print("captured:", frame.shape)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## Quality gate diagnostics

Runs the same checks `quality.assess_quality` uses in the real pipeline, but prints the actual numbers (blur variance, dark/bright pixel ratios, OCR character count) instead of just pass/fail -- useful for seeing *how close* a marginal shot is, not just whether it was accepted.

In [ ]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

is_sharp, variance = quality.check_blur(gray)
print(f"blur variance: {variance:.1f} (threshold {quality.BLUR_VARIANCE_THRESHOLD}) -> {'OK' if is_sharp else 'TOO BLURRY'}")

exposure_ok, reason = quality.check_exposure(gray)
print(f"exposure: {'OK' if exposure_ok else reason}")

text_ok, char_count = quality.check_text_presence(gray, ocr.run_ocr)
print(f"OCR text-presence: {char_count} chars found (need {quality.MIN_OCR_CHARS}) -> {'OK' if text_ok else 'NOT ENOUGH TEXT'}")

accepted, reason = quality.assess_quality(frame, ocr.run_ocr)
print()
print("Overall:", "ACCEPTED" if accepted else f"REJECTED -- {reason}")

## Run it through the real pipeline

Same `pipeline.process_image` the deployed app calls -- barcode/QR decode, OCR, field parsing, validation. Encodes the captured frame as JPEG bytes first, matching what the app receives from `st.camera_input()`/`st.file_uploader`.

In [ ]:
ok, buffer = cv2.imencode(".jpg", frame)
assert ok
image_bytes = buffer.tobytes()

result = pipeline.process_image(image_bytes, "camera_capture.jpg")
result

## Burst comparison

Capture several frames in a row (e.g. while trying different distances, angles, or lighting) and compare their quality metrics side by side. Re-run the capture cell between each entry, or adapt this into a loop with `time.sleep` between shots if you want to physically move the camera/tag between captures.

In [ ]:
# Each time you like a capture from above, append it here for comparison:
# burst.append(("label describing this shot", frame.copy()))
burst = []

In [ ]:
burst.append(("e.g. 30cm, ceiling light", frame.copy()))
print(f"{len(burst)} shot(s) in burst")

In [ ]:
for label, shot in burst:
    gray_shot = cv2.cvtColor(shot, cv2.COLOR_BGR2GRAY)
    _, variance = quality.check_blur(gray_shot)
    _, exposure_reason = quality.check_exposure(gray_shot)
    accepted, reason = quality.assess_quality(shot, ocr.run_ocr)
    ok, buf = cv2.imencode(".jpg", shot)
    parsed = pipeline.process_image(buf.tobytes(), label)
    print(f"=== {label} ===")
    print(f"  shape={shot.shape}  blur_variance={variance:.1f}  exposure={exposure_reason or 'ok'}")
    print(f"  quality gate: {'ACCEPTED' if accepted else f'REJECTED ({reason})'}")
    if parsed["accepted"]:
        print(f"  needs_review={parsed['needs_review']}  fields={ {k: parsed[k] for k in ['igi_report_no','report_type','shape','carat','color','clarity']} }")
    print()